In [0]:
df_silver = spark.read.table("workspace.`sports-pipeline`.football_matches_silver")
df_silver.show(10)

In [0]:
from pyspark.sql.functions import col, count, sum, round 

df_gold_wins = df_silver\
    .filter(col("results") != "Draw") \
    .filter(col("status") == "FINISHED") \
    .groupBy("results") \
    .agg(count("*").alias("total_wins"))\
    .orderBy(col("total_wins").desc())\
    .withColumnsRenamed({"results": "winning_team"})

df_gold_wins.show(10)

In [0]:
df_gold_goals = df_silver \
    .filter(col("status") == "FINISHED") \
    .groupBy("competition") \
    .agg(
        count("*").alias("total_matches"),
        sum("home_score").alias("home_goals"),
        sum("away_score").alias("away_goals"),
        (sum("home_score") + sum("away_score")).alias("total_goals"),
        round((sum("home_score") + sum("away_score")) / count("*"), 2).alias("avg_goals_per_match")
    ) \
    .orderBy(col("total_goals").desc())

df_gold_goals.show(10)

In [0]:
df_gold_results = df_silver \
    .groupBy("results") \
    .agg(count("*").alias("total")) \
    .orderBy(col("total").desc())

df_gold_results.show()

In [0]:
df_gold_wins.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.`sports-pipeline`.football_wins_gold")

df_gold_goals.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.`sports-pipeline`.football_goals_gold")

df_gold_results.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.`sports-pipeline`.football_results_gold")

print("Tablas Gold creadas!")